# Driver acceptance: what the 2022 tables tell us

This notebook is a reading companion to **Mohamed Fawaz Hussain Fareed's assessed BSc dissertation (2022)**. The first two sections display **historical reported aggregates**, and the final section generates **fabricated observations** to demonstrate statistical methods.

No respondent-level records are included; running this notebook **cannot reproduce the original SPSS execution**.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
assert (ROOT/"data/published_2022_aggregates/construct_reliability_2022.csv").exists()
SRC = ROOT / "data/published_2022_aggregates"
correlations = pd.read_csv(ROOT/"data/published_2022_correlations.csv")
correlations

## Associations are not interventions

The nine correlations below come from the **2022 submitted dissertation (N = 311)**. They describe reported survey associations, not estimates of accidents avoided.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
rows = correlations.iloc[::-1]
colors = ["#a35f50" if x < 0 else "#376c80" for x in rows.pearson_r]
ax.barh(rows.construct, rows.pearson_r, color=colors)
ax.axvline(0, color="#333", linewidth=.8)
ax.set(xlim=(-.4,.9), xlabel="Pearson r with driver acceptance")
ax.set_title("Historically reported pairwise associations, 2022")
fig.tight_layout()
plt.show()

## Why the simultaneous regression matters

A large pairwise correlation can be reduced after including overlapping predictors. The original dissertation used a **1% statistical significance threshold**. Under that decision rule, perceived usefulness with p = .011 is **not** a significant adjusted association. R² = .679 describes the **original model's fitted sample**, not external predictive performance.

In [ ]:
coefs = pd.read_csv(SRC/"multiple_regression_coefficients_2022.csv")
alphas = pd.read_csv(SRC/"construct_reliability_2022.csv")
print("HISTORICAL 2022 transcription. Not computed from synthetic data.")
print("Reported R² = 0.679 (in-sample, N = 311)")
display(coefs)
display(alphas)

## A separate, fabricated-data illustration (2026)

The cells below use generated numbers **unrelated to the questionnaire respondents**. The outputs cannot validate, replace, or reproduce the 2022 SPSS findings.

In [ ]:
import sys
import numpy as np
sys.path.insert(0,str(ROOT/"src"))
from bsc_public.stats import cronbach_alpha, ols
from bsc_public.factor_validity import pca_kmo_bartlett
rng=np.random.default_rng(2026)
shared=rng.normal(size=(400,1))
items=shared+rng.normal(scale=.55,size=(400,4))
X=np.column_stack([rng.normal(size=400) for _ in range(9)])
y=2+.5*X[:,0]-.3*X[:,1]+rng.normal(scale=.8,size=400)
reliability=cronbach_alpha(items)
pc=pca_kmo_bartlett(items)
fit=ols(X,y)
print("SYNTHETIC ONLY: 400 fabricated observations")
print("Synthetic Cronbach alpha:",round(reliability,3))
print("Synthetic KMO:",round(pc["kmo"],3))
print("Synthetic first-component variance %:",round(pc["first_component_variance_pct"],3))
print("Synthetic fitted-sample R²:",round(fit["r_squared_in_sample"],3))
assert len(y) != 311